# 안전성 검증 도구 - 라벨 매핑 안정성 / 화질 진단 / 고위험 혼동 쌍

약을 다루는 서비스라 오탐이 사용자 건강과 직결된다는 전제로, 지금 코드/데이터로 바로
검증 가능한 세 가지 위험을 도구로 만들었다.

1. **라벨 매핑 안정성**: category_id -> label 번호 매핑이 데이터가 조금만 바뀌어도
   조용히 틀어질 수 있는지 확인
2. **화질 진단**: 학습 데이터(AI-Hub 스튜디오 사진)에 블러/노출 문제가 있는 사진이
   섞여 있는지, 그리고 실제 사용 환경(흔들림/저조도)에서 성능이 얼마나 떨어질 수 있는지
3. **고위험 혼동 쌍**: 어떤 두 약이 시각적으로 가장 비슷해서 모델이 헷갈릴 위험이
   가장 큰지 순위를 매겨서 뽑아낸다 (약사/멘토님께 검토 요청할 리스트로 바로 쓸 수 있음)

## STEP 0 : 환경설정

In [ ]:
#@title (1) Import Module
import os
import re
import json
import glob
import random
from collections import defaultdict, Counter

import numpy as np
import cv2
import torch
import torchvision
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print("device:", device)


In [ ]:
#@title (2) 데이터 경로 + 정제된 annotation 불러오기
DATA_ROOT = "/Users/codeit/Downloads/sprint_ai_project1_data"
TRAIN_IMAGE_DIR = os.path.join(DATA_ROOT, "train_images")

tightened_path = os.path.join(DATA_ROOT, "tightened_annotations.json")
with open(tightened_path, "r", encoding="utf-8") as f:
    image_records = json.load(f)

category_id_to_name = {}
for rec in image_records.values():
    for ann in rec["annotations"]:
        category_id_to_name.setdefault(ann["category_id"], None)

# 이름은 원본 annotation에서 한 번만 다시 채워온다 (tightened json엔 이름을 안 담아서)
annotation_paths = glob.glob(os.path.join(DATA_ROOT, "train_annotations", "**", "*.json"), recursive=True)
for p in annotation_paths:
    d = json.load(open(p, encoding="utf-8"))
    img = d["images"][0]
    m = re.fullmatch(r"K-(\d+)", str(img["dl_mapping_code"]).strip())
    if m:
        cid = int(m.group(1))
        if cid in category_id_to_name and category_id_to_name[cid] is None:
            category_id_to_name[cid] = img["dl_name"]

print(f"이미지 {len(image_records)}장, 클래스 {len(category_id_to_name)}종 로드 완료")


## 도구 1 : 라벨 매핑 안정성 검증

`category_to_label = {cid: i+1 for i, cid in enumerate(sorted(category_id_to_name))}` 방식은
"지금 이 순간 데이터에 어떤 클래스가 있는지"에 따라 번호가 정해진다. 데이터가 바뀌면
(이번처럼 충돌 annotation 30개를 제외하는 것만으로도) 번호가 밀릴 수 있는지 실제로
두 가지 시나리오를 만들어서 대조해본다.

In [ ]:
#@title (1-A) 실제로 라벨 번호가 밀리는지 재현
# 시나리오 A: 지금 데이터(정제 후, 3,731장) 그대로
sorted_ids_A = sorted(category_id_to_name.keys())
category_to_label_A = {cid: i + 1 for i, cid in enumerate(sorted_ids_A)}

# 시나리오 B: 나중에 AI-Hub 데이터를 조금 더 받아서 새로운 클래스 하나가 추가로 섞여
# 들어왔다고 가정 (실제로 이런 일이 매우 흔함 - 팀이 AI-Hub 데이터를 계속 늘려왔던 것처럼)
sorted_ids_B = sorted(list(category_id_to_name.keys()) + [1])  # 맨 앞에 새 클래스 하나 추가 (예: id=1)
category_to_label_B = {cid: i + 1 for i, cid in enumerate(sorted_ids_B)}

shifted = [cid for cid in sorted_ids_A if category_to_label_A[cid] != category_to_label_B[cid]]

print(f"시나리오 A 클래스 수: {len(sorted_ids_A)}, 시나리오 B 클래스 수: {len(sorted_ids_B)}")
print(f"클래스 하나만 추가됐는데 label 번호가 바뀐 기존 클래스 수: {len(shifted)} / {len(sorted_ids_A)}")
if shifted:
    example = shifted[len(shifted)//2]
    print(f"예시: {category_id_to_name[example]} (category_id={example})는 "
          f"기존에 label {category_to_label_A[example]}였는데 새 데이터가 섞이면 label {category_to_label_B[example]}로 바뀜")
    print("=> 이 상태로 예전 체크포인트를 새 label 매핑으로 추론하면, 모델 성능과 무관하게")
    print("   예측 결과가 전부 다른 약으로 조용히 밀려서 나온다 (에러 없이 틀림 - 제일 위험한 유형).")


In [ ]:
#@title (1-B) 해결책: 라벨 매핑을 파일로 고정해서 버전 관리
# 학습마다 다시 계산하지 말고, 한 번 정한 매핑을 파일로 저장해서 학습/추론 양쪽이
# 항상 "같은 파일"을 참조하게 만든다. 새 클래스는 기존 번호를 안 건드리고 맨 뒤에 추가.
LABEL_MAP_PATH = os.path.join(DATA_ROOT, "category_label_map.json")


def load_or_create_label_map(category_id_to_name, path=LABEL_MAP_PATH):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            saved = json.load(f)
        category_to_label = {int(k): v for k, v in saved["category_to_label"].items()}
        next_label = saved["next_label"]

        new_cids = [cid for cid in category_id_to_name if cid not in category_to_label]
        for cid in sorted(new_cids):
            category_to_label[cid] = next_label
            next_label += 1
        if new_cids:
            print(f"새로 등장한 클래스 {len(new_cids)}개를 기존 매핑 뒤에 추가함: {new_cids}")
    else:
        sorted_ids = sorted(category_id_to_name.keys())
        category_to_label = {cid: i + 1 for i, cid in enumerate(sorted_ids)}
        next_label = len(sorted_ids) + 1
        print("기존 매핑 파일이 없어서 새로 생성함 (이후로는 이 파일이 기준이 됨)")

    with open(path, "w", encoding="utf-8") as f:
        json.dump({"category_to_label": category_to_label, "next_label": next_label}, f, ensure_ascii=False, indent=2)

    return category_to_label


category_to_label = load_or_create_label_map(category_id_to_name)
label_to_category = {v: k for k, v in category_to_label.items()}
print(f"고정된 매핑 저장 완료: {LABEL_MAP_PATH}")
print("앞으로 모든 학습/추론 노트북은 sorted()로 다시 계산하지 말고 이 파일을 불러써야 한다.")


## 도구 2 : 화질 진단

학습 데이터 자체의 화질 분포를 재보고(블러/노출/대비), 실제 사용자가 흔들리거나
어두운 곳에서 찍을 때 얼마나 나쁜 조건까지 가정해야 하는지 감을 잡는다.

In [ ]:
#@title (2-A) 학습 데이터 화질 프로파일링 (블러/노출/대비)
def blur_score(gray):
    return cv2.Laplacian(gray, cv2.CV_64F).var()  # 낮을수록 흐릿함


def quality_stats(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return {
        "blur": blur_score(gray),
        "brightness": gray.mean(),
        "contrast": gray.std(),
        "overexposed_ratio": (gray > 250).mean(),
        "underexposed_ratio": (gray < 5).mean(),
    }


sample_stems = random.sample(list(image_records.keys()), min(800, len(image_records)))
rows = []
corrupted = []
for stem in tqdm(sample_stems, desc="화질 측정 중"):
    path = os.path.join(TRAIN_IMAGE_DIR, image_records[stem]["file_name"])
    stats = quality_stats(path)
    if stats is None:
        corrupted.append(stem)
        continue
    stats["stem"] = stem
    rows.append(stats)

import pandas as pd
quality_df = pd.DataFrame(rows)
print(f"측정 실패(파일 손상/열기 실패): {len(corrupted)}개")
print(quality_df[["blur", "brightness", "contrast", "overexposed_ratio", "underexposed_ratio"]].describe())

blur_p5 = quality_df["blur"].quantile(0.05)
suspect_blurry = quality_df[quality_df["blur"] <= blur_p5].sort_values("blur")
print(f"\n블러 하위 5% (가장 흐릿한 학습 이미지 후보, {len(suspect_blurry)}장):")
print(suspect_blurry[["stem", "blur"]].head(10).to_string(index=False))


In [ ]:
#@title (2-B) 실제 사용 환경 시뮬레이션 - 인위적으로 열화시켜서 얼마나 나빠지는지 확인
# 학습 데이터는 스튜디오 사진이라 블러/저조도/압축이 거의 없다. 실제 사용자 사진은
# 다를 거라는 가정 하에, 대표 이미지 몇 장에 인위적으로 열화를 줘서 시각적으로
# "이 정도까지 나빠지면 각인이 안 보이는구나"를 미리 확인해둔다.
def simulate_motion_blur(img, k=15):
    kernel = np.zeros((k, k))
    kernel[k // 2, :] = 1.0 / k
    return cv2.filter2D(img, -1, kernel)


def simulate_low_light(img, factor=0.35):
    return np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)


def simulate_jpeg_compression(img, quality=15):
    ok, enc = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return cv2.imdecode(enc, cv2.IMREAD_COLOR)


def simulate_noise(img, sigma=20):
    noise = np.random.normal(0, sigma, img.shape)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)


demo_stem = random.choice(list(image_records.keys()))
demo_img = cv2.imread(os.path.join(TRAIN_IMAGE_DIR, image_records[demo_stem]["file_name"]))

variants = {
    "원본 (학습 데이터 그대로)": demo_img,
    "모션 블러 (손떨림 가정)": simulate_motion_blur(demo_img),
    "저조도 (어두운 방 가정)": simulate_low_light(demo_img),
    "JPEG 저품질 압축": simulate_jpeg_compression(demo_img),
    "센서 노이즈 (저가 카메라 가정)": simulate_noise(demo_img),
}

fig, axes = plt.subplots(1, len(variants), figsize=(4 * len(variants), 4))
for ax, (label, img) in zip(axes, variants.items()):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ax.set_title(f"{label}\nblur={blur_score(gray):.0f}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("=> 이 5가지 조건을 실제 augmentation 파이프라인에 넣어서 학습해야, 실사용 환경에서도")
print("   모델이 급격히 무너지지 않는다. 지금 학습 노트북들의 train_transformer에는")
print("   RandomHorizontalFlip/Rotation/ColorJitter만 있고 블러·저조도·압축·노이즈가 없다.")


## 도구 3 : 고위험 혼동 쌍 (시각적으로 가장 비슷한 클래스 순위)

클래스별 임베딩 간 코사인 유사도를 계산해서, "모델이 헷갈릴 가능성이 가장 높은 두 약"
순위를 뽑는다. 인스턴스 개수와 무관하게 "생김새 자체가 위험하게 비슷한 쌍"을 찾는 게
목적이라, 이 리스트는 팀 내부 판단보다 약사/멘토님 검토를 받는 게 정확하다.

In [ ]:
#@title (3-A) 클래스별 임베딩 추출 (사전학습 ResNet18)
embed_backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
embed_backbone.fc = torch.nn.Identity()
embed_backbone.eval().to(device)

embed_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((96, 96)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def crops_for_class(category_id, max_crops=15):
    crops = []
    for stem, record in image_records.items():
        for ann in record["annotations"]:
            if ann["category_id"] != category_id:
                continue
            x, y, w, h = [int(round(v)) for v in ann["bbox"]]
            img_path = os.path.join(TRAIN_IMAGE_DIR, record["file_name"])
            with Image.open(img_path) as im:
                crop = im.convert("RGB").crop((x, y, x + w, y + h))
            crops.append(embed_transform(crop))
            if len(crops) >= max_crops:
                return crops
    return crops


class_embeddings = {}
with torch.inference_mode():
    for cid, name in tqdm(category_id_to_name.items(), desc="클래스 임베딩 추출"):
        crops = crops_for_class(cid)
        if not crops:
            continue
        feats = embed_backbone(torch.stack(crops).to(device))
        feats = torch.nn.functional.normalize(feats, dim=1)  # 코사인 유사도용 정규화
        class_embeddings[cid] = feats.mean(dim=0).cpu().numpy()

print(f"임베딩 추출된 클래스 수: {len(class_embeddings)}")


In [ ]:
#@title (3-B) 가장 헷갈리기 쉬운 클래스 쌍 순위 뽑기
cids = list(class_embeddings.keys())
emb_matrix = np.stack([class_embeddings[c] for c in cids])
emb_matrix = emb_matrix / (np.linalg.norm(emb_matrix, axis=1, keepdims=True) + 1e-8)

sim_matrix = emb_matrix @ emb_matrix.T

pairs = []
for i in range(len(cids)):
    for j in range(i + 1, len(cids)):
        pairs.append((cids[i], cids[j], sim_matrix[i, j]))

pairs.sort(key=lambda x: -x[2])

print("시각적으로 가장 비슷한 클래스 쌍 TOP 15 (코사인 유사도 1.0에 가까울수록 위험):")
for cid_a, cid_b, sim in pairs[:15]:
    n_a = class_instance_count = sum(1 for r in image_records.values() for a in r["annotations"] if a["category_id"] == cid_a)
    n_b = sum(1 for r in image_records.values() for a in r["annotations"] if a["category_id"] == cid_b)
    print(f"  {sim:.3f}  {category_id_to_name[cid_a]:<28} (n={n_a:>3})  <->  "
          f"{category_id_to_name[cid_b]:<28} (n={n_b:>3})")

print("\n이 리스트에서 유사도 0.9 이상인 쌍은 이미지 몇 장씩 나란히 놓고 멘토님/약사님께")
print("실제로 헷갈릴 만한 쌍인지 확인받는 걸 권장. 확인되면 해당 두 클래스는 threshold를")
print("따로 관리하거나, 두 클래스만 따로 모아 추가 학습(hard example mining)하는 것도 방법.")


In [ ]:
#@title (3-C) 상위 위험 쌍 이미지 나란히 비교
def show_pair_comparison(cid_a, cid_b, n=3):
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    for row, cid in enumerate([cid_a, cid_b]):
        stems_with_class = [s for s, r in image_records.items() if any(a["category_id"] == cid for a in r["annotations"])]
        picked = random.sample(stems_with_class, min(n, len(stems_with_class)))
        for col, stem in enumerate(picked):
            record = image_records[stem]
            img = cv2.cvtColor(cv2.imread(os.path.join(TRAIN_IMAGE_DIR, record["file_name"])), cv2.COLOR_BGR2RGB)
            ann = next(a for a in record["annotations"] if a["category_id"] == cid)
            x, y, w, h = [int(v) for v in ann["bbox"]]
            crop = img[max(0,y-10):y+h+10, max(0,x-10):x+w+10]
            axes[row, col].imshow(crop)
            axes[row, col].axis("off")
        axes[row, 0].set_ylabel(category_id_to_name[cid], fontsize=10)
    plt.suptitle(f"{category_id_to_name[cid_a]}  vs  {category_id_to_name[cid_b]}")
    plt.tight_layout()
    plt.show()


# 가장 위험한 1위 쌍을 바로 확인
if pairs:
    show_pair_comparison(pairs[0][0], pairs[0][1])
